### Data ingestion

In [0]:
raw = spark \
      .read \
      .csv("/Volumes/workspace/ecommerce/ecommerce_data_sample/sample_df.csv", header=True, inferSchema=True)


In [0]:
from pyspark.sql import functions as F
raw.withColumn("ingestion_time", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").save("/Volumes/workspace/ecommerce/delta/bronze/events")

In [0]:
bronze = spark.read.format("delta").load("/Volumes/workspace/ecommerce/delta/bronze/events")

### Task 1: Create User-Level Feature Table

In [0]:
from pyspark.sql import functions as F
def get_final_category(category_code):
    if category_code is None:
        return None
    return category_code.split(".")[-1]

silver = df.filter(F.col('price') > 0) \
  .dropDuplicates(['user_session','event_time']) \
  .filter(F.col('category_code').isNotNull()) \
  .withColumn("product_category", F.udf(get_final_category)(F.col('category_code'))) \
  .withColumn("price_tier",
                         F.when(F.col("price") < 100, "budget")
                          .when(F.col("price") < 200, "affordable")
                          .when(F.col("price") < 500, "midrange")
                          .when(F.col("price") < 1000, "luxury")
                          .otherwise("ultra_luxury")) 


In [0]:
from pyspark.sql import functions as F

features_df = silver.filter(F.col("event_type")=="purchase") \
    .groupBy('product_category').agg(
    F.count("*").alias("total_items"),
    F.round(F.sum("price"),2).alias("total_revenue"),
    F.round(F.avg("price"),2).alias("avg_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price")
)
    
features_df.show()

### Task 2: Save as Delta (Silver Layer)

In [0]:
features_df.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/ecommerce/delta/silver/user_features")

### Task 3: Ensure No Duplicates

In [0]:
features_df = features_df.dropDuplicates(['product_category'])

### Task 4: Validate Feature Quality

In [0]:
features_df.filter("total_revenue IS NULL").count()

In [0]:
features_df.filter("total_revenue < 0").show()